In [1]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "MNIST"  # Options: "MNIST", "FashionMNIST"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "active_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# ---------------------------
# Model - Updated to take num_classes parameter
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)  # Uses parameter

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both MNIST and FashionMNIST have 10 classes

def load_dataset(dataset_name, train=True):
    """Load MNIST or FashionMNIST datasets"""
    dataset_name = dataset_name.upper()

    if dataset_name == "MNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        dataset = datasets.MNIST(root="./data", train=train, download=True, transform=transform)

    elif dataset_name == "FASHIONMNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
        dataset = datasets.FashionMNIST(root="./data", train=train, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only MNIST and FashionMNIST are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(600, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Provide dataset source for the trained model
                row[fn] = "mnist"  # Use "mnist" for both MNIST and FashionMNIST
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED)
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            if trigger_pos == "bottom_right":
                arr[0, -BACKDOOR_TRIGGER_SIZE:, -BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[0, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start = 14 - BACKDOOR_TRIGGER_SIZE//2
                arr[0, start:start+BACKDOOR_TRIGGER_SIZE, start:start+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")
    print("=" * 70)

    # Get number of classes for this dataset
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct number of classes
    def model_factory():
        return SimpleCNN(num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - MNIST
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset: MNIST, Number of classes: 10
Loaded detector pipeline from: active_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']


100%|██████████| 9.91M/9.91M [00:00<00:00, 19.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 510kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.71MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.02MB/s]



Client-Class Distribution:
  Client 0: 600 samples, class(es): [7]
  Client 1: 600 samples, class(es): [3]
  Client 2: 600 samples, class(es): [2]
  Client 3: 600 samples, class(es): [8]
  Client 4: 600 samples, class(es): [5]
  Client 5: 600 samples, class(es): [6]
  Client 6: 600 samples, class(es): [9]
  Client 7: 600 samples, class(es): [4]
  Client 8: 600 samples, class(es): [0]
  Client 9: 600 samples, class(es): [1]

RUNNING CLEAN BASELINE (no attacks)
    Clean Baseline Round 10 - Global Accuracy: 0.6312
    Clean Baseline Round 20 - Global Accuracy: 0.7163
    Clean Baseline Round 30 - Global Accuracy: 0.7588
    Clean Baseline Round 40 - Global Accuracy: 0.7900
    Clean Baseline Round 50 - Global Accuracy: 0.7412

Clean Baseline Final Global Accuracy: 0.7438
Saved clean baseline global model state: global_model_clean_baseline_mnist.pt

RUNNING PARALLEL FL WITH ATTACKS

ROUND 1/50
Ground-truth malicious mapping (client_id -> attack_type): {5: 5, 4: 2, 8: 4}
Client  0 | GT=CL

In [2]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "FashionMNIST"  # Options: "MNIST", "FashionMNIST"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "active_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# ---------------------------
# Model - Updated to take num_classes parameter
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)  # Uses parameter

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both MNIST and FashionMNIST have 10 classes

def load_dataset(dataset_name, train=True):
    """Load MNIST or FashionMNIST datasets"""
    dataset_name = dataset_name.upper()

    if dataset_name == "MNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        dataset = datasets.MNIST(root="./data", train=train, download=True, transform=transform)

    elif dataset_name == "FASHIONMNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
        dataset = datasets.FashionMNIST(root="./data", train=train, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only MNIST and FashionMNIST are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(600, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Provide dataset source for the trained model
                row[fn] = "mnist"  # Use "mnist" for both MNIST and FashionMNIST
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED)
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            if trigger_pos == "bottom_right":
                arr[0, -BACKDOOR_TRIGGER_SIZE:, -BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[0, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start = 14 - BACKDOOR_TRIGGER_SIZE//2
                arr[0, start:start+BACKDOOR_TRIGGER_SIZE, start:start+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")
    print("=" * 70)

    # Get number of classes for this dataset
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct number of classes
    def model_factory():
        return SimpleCNN(num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - FashionMNIST
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset: FashionMNIST, Number of classes: 10
Loaded detector pipeline from: active_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']


100%|██████████| 26.4M/26.4M [00:02<00:00, 9.50MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 204kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.76MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.66MB/s]



Client-Class Distribution:
  Client 0: 600 samples, class(es): [7]
  Client 1: 600 samples, class(es): [3]
  Client 2: 600 samples, class(es): [2]
  Client 3: 600 samples, class(es): [8]
  Client 4: 600 samples, class(es): [5]
  Client 5: 600 samples, class(es): [6]
  Client 6: 600 samples, class(es): [9]
  Client 7: 600 samples, class(es): [4]
  Client 8: 600 samples, class(es): [0]
  Client 9: 600 samples, class(es): [1]

RUNNING CLEAN BASELINE (no attacks)
    Clean Baseline Round 10 - Global Accuracy: 0.5225
    Clean Baseline Round 20 - Global Accuracy: 0.6075
    Clean Baseline Round 30 - Global Accuracy: 0.6275
    Clean Baseline Round 40 - Global Accuracy: 0.6637
    Clean Baseline Round 50 - Global Accuracy: 0.6813

Clean Baseline Final Global Accuracy: 0.7188
Saved clean baseline global model state: global_model_clean_baseline_fashionmnist.pt

RUNNING PARALLEL FL WITH ATTACKS

ROUND 1/50
Ground-truth malicious mapping (client_id -> attack_type): {6: 1, 9: 3, 5: 2}
Client  0 

general classifier


In [3]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "MNIST"  # Options: "MNIST", "FashionMNIST"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "combined_fl_attack_detector_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# ---------------------------
# Model - Updated to take num_classes parameter
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)  # Uses parameter

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both MNIST and FashionMNIST have 10 classes

def load_dataset(dataset_name, train=True):
    """Load MNIST or FashionMNIST datasets"""
    dataset_name = dataset_name.upper()

    if dataset_name == "MNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        dataset = datasets.MNIST(root="./data", train=train, download=True, transform=transform)

    elif dataset_name == "FASHIONMNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
        dataset = datasets.FashionMNIST(root="./data", train=train, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only MNIST and FashionMNIST are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(600, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Provide dataset source for the trained model
                row[fn] = "mnist"  # Use "mnist" for both MNIST and FashionMNIST
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED)
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            if trigger_pos == "bottom_right":
                arr[0, -BACKDOOR_TRIGGER_SIZE:, -BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[0, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start = 14 - BACKDOOR_TRIGGER_SIZE//2
                arr[0, start:start+BACKDOOR_TRIGGER_SIZE, start:start+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")
    print("=" * 70)

    # Get number of classes for this dataset
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct number of classes
    def model_factory():
        return SimpleCNN(num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - MNIST
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset: MNIST, Number of classes: 10
Loaded detector pipeline from: combined_fl_attack_detector_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']

Client-Class Distribution:
  Client 0: 600 samples, class(es): [7]
  Client 1: 600 samples, class(es): [3]
  Client 2: 600 samples, class(es): [2]
  Client 3: 600 samples, class(es): [8]
  Client 4: 600 samples, class(es): [5]
  Client 5: 600 samples, class(es): [6]
  Client 6: 600 samples, class(es): [9]
  Client 7: 600 samples, class(es): [4]
  Client 8: 600 samples, class(es): [0]
  Client 9: 600 samples, class(es): [1]

RUNNING CLEAN BASELINE (no attacks)
    Clean Baseline Round 10 - Global Accuracy: 0.6312
    Clean Baseline Roun

In [4]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "FashionMNIST"  # Options: "MNIST", "FashionMNIST"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "combined_fl_attack_detector_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# ---------------------------
# Model - Updated to take num_classes parameter
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)  # Uses parameter

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both MNIST and FashionMNIST have 10 classes

def load_dataset(dataset_name, train=True):
    """Load MNIST or FashionMNIST datasets"""
    dataset_name = dataset_name.upper()

    if dataset_name == "MNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        dataset = datasets.MNIST(root="./data", train=train, download=True, transform=transform)

    elif dataset_name == "FASHIONMNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.2860,), (0.3530,))
        ])
        dataset = datasets.FashionMNIST(root="./data", train=train, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only MNIST and FashionMNIST are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(600, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Provide dataset source for the trained model
                row[fn] = "mnist"  # Use "mnist" for both MNIST and FashionMNIST
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED)
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            if trigger_pos == "bottom_right":
                arr[0, -BACKDOOR_TRIGGER_SIZE:, -BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[0, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start = 14 - BACKDOOR_TRIGGER_SIZE//2
                arr[0, start:start+BACKDOOR_TRIGGER_SIZE, start:start+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")
    print("=" * 70)

    # Get number of classes for this dataset
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > 80:
            test_subset_indices.extend(random.sample(indices, 80))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct number of classes
    def model_factory():
        return SimpleCNN(num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - FashionMNIST
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset: FashionMNIST, Number of classes: 10
Loaded detector pipeline from: combined_fl_attack_detector_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']

Client-Class Distribution:
  Client 0: 600 samples, class(es): [7]
  Client 1: 600 samples, class(es): [3]
  Client 2: 600 samples, class(es): [2]
  Client 3: 600 samples, class(es): [8]
  Client 4: 600 samples, class(es): [5]
  Client 5: 600 samples, class(es): [6]
  Client 6: 600 samples, class(es): [9]
  Client 7: 600 samples, class(es): [4]
  Client 8: 600 samples, class(es): [0]
  Client 9: 600 samples, class(es): [1]

RUNNING CLEAN BASELINE (no attacks)
    Clean Baseline Round 10 - Global Accuracy: 0.5225
    Clean

CIFAR10

In [5]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "CIFAR10"  # Options: "CIFAR10", "SVHN"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "active_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# Dataset configurations: (channels, height, width) and normalization stats
DATASET_CONFIGS = {
    "SVHN": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4377, 0.4438, 0.4728),
        "std": (0.1980, 0.2010, 0.1970)
    },
    "CIFAR10": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4914, 0.4822, 0.4465),
        "std": (0.2023, 0.1994, 0.2010)
    }
}

# ---------------------------
# Model - Flexible CNN for different dataset sizes and channels
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, input_size=(28, 28), num_classes=10):
        """
        Flexible CNN that adapts to different input sizes and channels

        Args:
            input_channels: Number of input channels (1 for grayscale, 3 for RGB)
            input_size: Tuple of (height, width)
            num_classes: Number of output classes
        """
        super(SimpleCNN, self).__init__()
        self.input_channels = input_channels
        self.input_size = input_size

        # First convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Third convolutional layer (added for better feature extraction in color images)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)

        # Calculate the size after three pooling operations
        h, w = input_size
        h = h // 8  # After three pool operations (each halves the size)
        w = w // 8

        # Calculate flattened size
        self.flattened_size = 128 * h * w

        # Fully connected layers
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        print(f"Model configured for input: {input_channels} channels, size {input_size}")
        print(f"Flattened size after conv layers: {self.flattened_size}")

    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = x.view(-1, self.flattened_size)

        # Fully connected layers with dropout
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)

        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both CIFAR10 and SVHN have 10 classes

def get_dataset_config(dataset_name):
    """Get configuration for each dataset"""
    dataset_name = dataset_name.upper()
    return DATASET_CONFIGS.get(dataset_name, DATASET_CONFIGS["CIFAR10"])

def load_dataset(dataset_name, train=True):
    """Load CIFAR10 or SVHN datasets"""
    dataset_name = dataset_name.upper()
    config = get_dataset_config(dataset_name)

    if dataset_name == "SVHN":
        # SVHN dataset - keep as RGB
        split = 'train' if train else 'test'

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        dataset = datasets.SVHN(root="./data", split=split, download=True, transform=transform)

    elif dataset_name == "CIFAR10":
        # CIFAR-10 dataset - keep as RGB
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        if train:
            dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
        else:
            dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only CIFAR10 and SVHN are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(1000, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Set dataset_source based on current dataset
                if DATASET == "CIFAR10":
                    row[fn] = "cifar"
                elif DATASET == "SVHN":
                    row[fn] = "cifar"  # Use "cifar" for both since your model was trained with "cifar"
                else:
                    row[fn] = "cifar"  # default fallback
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED) - Modified for variable input sizes and channels
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            channels, h, w = arr.shape

            if trigger_pos == "bottom_right":
                # Apply trigger to all channels for RGB
                arr[:, h-BACKDOOR_TRIGGER_SIZE:, w-BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[:, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start_h = h // 2 - BACKDOOR_TRIGGER_SIZE // 2
                start_w = w // 2 - BACKDOOR_TRIGGER_SIZE // 2
                arr[:, start_h:start_h+BACKDOOR_TRIGGER_SIZE, start_w:start_w+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        test_samples = min(100, len(indices))

        if len(indices) > test_samples:
            test_subset_indices.extend(random.sample(indices, test_samples))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")

    # Get dataset configuration
    config = get_dataset_config(DATASET)
    channels = config["channels"]
    input_size = config["size"]
    color_mode = "RGB"

    print(f"Dataset Mode: {color_mode}, Input: {channels} channels, size {input_size}")
    print("=" * 70)

    # Get dataset properties
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}, Input: {channels} channels, size {input_size}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    # Check dataset sizes
    sample_img, sample_label = train_dataset[0]
    print(f"\nDataset loaded successfully:")
    print(f"  Sample image shape: {sample_img.shape}")
    print(f"  Sample label: {sample_label}")
    print(f"  Expected input: {channels} channels, size {input_size}")

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    test_samples_per_class = 100

    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > test_samples_per_class:
            test_subset_indices.extend(random.sample(indices, test_samples_per_class))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct input channels, size, and number of classes
    def model_factory():
        return SimpleCNN(input_channels=channels, input_size=input_size, num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis but not for classification
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - CIFAR10
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset Mode: RGB, Input: 3 channels, size (32, 32)
Dataset: CIFAR10, Number of classes: 10, Input: 3 channels, size (32, 32)
Loaded detector pipeline from: active_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']


100%|██████████| 170M/170M [00:04<00:00, 36.1MB/s]



Dataset loaded successfully:
  Sample image shape: torch.Size([3, 32, 32])
  Sample label: 6
  Expected input: 3 channels, size (32, 32)

Client-Class Distribution:
  Client 0: 1000 samples, class(es): [7]
  Client 1: 1000 samples, class(es): [3]
  Client 2: 1000 samples, class(es): [2]
  Client 3: 1000 samples, class(es): [8]
  Client 4: 1000 samples, class(es): [5]
  Client 5: 1000 samples, class(es): [6]
  Client 6: 1000 samples, class(es): [9]
  Client 7: 1000 samples, class(es): [4]
  Client 8: 1000 samples, class(es): [0]
  Client 9: 1000 samples, class(es): [1]

RUNNING CLEAN BASELINE (no attacks)
Model configured for input: 3 channels, size (32, 32)
Flattened size after conv layers: 2048
Model configured for input: 3 channels, size (32, 32)
Flattened size after conv layers: 2048
    Clean Baseline Round 10 - Global Accuracy: 0.1370
    Clean Baseline Round 20 - Global Accuracy: 0.1220
    Clean Baseline Round 30 - Global Accuracy: 0.1080
    Clean Baseline Round 40 - Global Ac

/tmp/ipython-input-1101500629.py:416: RuntimeWarning: invalid value encountered in subtract
  parts.append(b - a)


Client  0 | GT=SCALE_NOISE  | variant=scale_and_noise                | pred=SCALE_NOISE  | FILTERED
Client  1 | GT=RANDOM_GAUSS | variant=random_gaussian                | pred=RANDOM_GAUSS | FILTERED
Client  2 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  3 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  4 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  5 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  6 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  7 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  8 | GT=BACKDOOR     | variant=backdoor_0_to_8_center         | pred=BACKDOOR     | FILTERED
Client  9 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Round 24 aggregated clients:

In [6]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "SVHN"  # Options: "CIFAR10", "SVHN"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "active_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# Dataset configurations: (channels, height, width) and normalization stats
DATASET_CONFIGS = {
    "SVHN": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4377, 0.4438, 0.4728),
        "std": (0.1980, 0.2010, 0.1970)
    },
    "CIFAR10": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4914, 0.4822, 0.4465),
        "std": (0.2023, 0.1994, 0.2010)
    }
}

# ---------------------------
# Model - Flexible CNN for different dataset sizes and channels
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, input_size=(28, 28), num_classes=10):
        """
        Flexible CNN that adapts to different input sizes and channels

        Args:
            input_channels: Number of input channels (1 for grayscale, 3 for RGB)
            input_size: Tuple of (height, width)
            num_classes: Number of output classes
        """
        super(SimpleCNN, self).__init__()
        self.input_channels = input_channels
        self.input_size = input_size

        # First convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Third convolutional layer (added for better feature extraction in color images)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)

        # Calculate the size after three pooling operations
        h, w = input_size
        h = h // 8  # After three pool operations (each halves the size)
        w = w // 8

        # Calculate flattened size
        self.flattened_size = 128 * h * w

        # Fully connected layers
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        print(f"Model configured for input: {input_channels} channels, size {input_size}")
        print(f"Flattened size after conv layers: {self.flattened_size}")

    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = x.view(-1, self.flattened_size)

        # Fully connected layers with dropout
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)

        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both CIFAR10 and SVHN have 10 classes

def get_dataset_config(dataset_name):
    """Get configuration for each dataset"""
    dataset_name = dataset_name.upper()
    return DATASET_CONFIGS.get(dataset_name, DATASET_CONFIGS["CIFAR10"])

def load_dataset(dataset_name, train=True):
    """Load CIFAR10 or SVHN datasets"""
    dataset_name = dataset_name.upper()
    config = get_dataset_config(dataset_name)

    if dataset_name == "SVHN":
        # SVHN dataset - keep as RGB
        split = 'train' if train else 'test'

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        dataset = datasets.SVHN(root="./data", split=split, download=True, transform=transform)

    elif dataset_name == "CIFAR10":
        # CIFAR-10 dataset - keep as RGB
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        if train:
            dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
        else:
            dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only CIFAR10 and SVHN are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(1000, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Set dataset_source based on current dataset
                if DATASET == "CIFAR10":
                    row[fn] = "cifar"
                elif DATASET == "SVHN":
                    row[fn] = "cifar"  # Use "cifar" for both since your model was trained with "cifar"
                else:
                    row[fn] = "cifar"  # default fallback
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED) - Modified for variable input sizes and channels
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            channels, h, w = arr.shape

            if trigger_pos == "bottom_right":
                # Apply trigger to all channels for RGB
                arr[:, h-BACKDOOR_TRIGGER_SIZE:, w-BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[:, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start_h = h // 2 - BACKDOOR_TRIGGER_SIZE // 2
                start_w = w // 2 - BACKDOOR_TRIGGER_SIZE // 2
                arr[:, start_h:start_h+BACKDOOR_TRIGGER_SIZE, start_w:start_w+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        test_samples = min(100, len(indices))

        if len(indices) > test_samples:
            test_subset_indices.extend(random.sample(indices, test_samples))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")

    # Get dataset configuration
    config = get_dataset_config(DATASET)
    channels = config["channels"]
    input_size = config["size"]
    color_mode = "RGB"

    print(f"Dataset Mode: {color_mode}, Input: {channels} channels, size {input_size}")
    print("=" * 70)

    # Get dataset properties
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}, Input: {channels} channels, size {input_size}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    # Check dataset sizes
    sample_img, sample_label = train_dataset[0]
    print(f"\nDataset loaded successfully:")
    print(f"  Sample image shape: {sample_img.shape}")
    print(f"  Sample label: {sample_label}")
    print(f"  Expected input: {channels} channels, size {input_size}")

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    test_samples_per_class = 100

    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > test_samples_per_class:
            test_subset_indices.extend(random.sample(indices, test_samples_per_class))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct input channels, size, and number of classes
    def model_factory():
        return SimpleCNN(input_channels=channels, input_size=input_size, num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis but not for classification
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - SVHN
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset Mode: RGB, Input: 3 channels, size (32, 32)
Dataset: SVHN, Number of classes: 10, Input: 3 channels, size (32, 32)
Loaded detector pipeline from: active_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']


100%|██████████| 182M/182M [00:01<00:00, 96.1MB/s]
100%|██████████| 64.3M/64.3M [00:01<00:00, 59.5MB/s]



Dataset loaded successfully:
  Sample image shape: torch.Size([3, 32, 32])
  Sample label: 1
  Expected input: 3 channels, size (32, 32)

Client-Class Distribution:
  Client 0: 1000 samples, class(es): [7]
  Client 1: 1000 samples, class(es): [3]
  Client 2: 1000 samples, class(es): [2]
  Client 3: 1000 samples, class(es): [8]
  Client 4: 1000 samples, class(es): [5]
  Client 5: 1000 samples, class(es): [6]
  Client 6: 1000 samples, class(es): [9]
  Client 7: 1000 samples, class(es): [4]
  Client 8: 1000 samples, class(es): [0]
  Client 9: 1000 samples, class(es): [1]

RUNNING CLEAN BASELINE (no attacks)
Model configured for input: 3 channels, size (32, 32)
Flattened size after conv layers: 2048
Model configured for input: 3 channels, size (32, 32)
Flattened size after conv layers: 2048
    Clean Baseline Round 10 - Global Accuracy: 0.1010
    Clean Baseline Round 20 - Global Accuracy: 0.1130
    Clean Baseline Round 30 - Global Accuracy: 0.1010
    Clean Baseline Round 40 - Global Ac

genearalised cifar

In [7]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "CIFAR10"  # Options: "CIFAR10", "SVHN"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "combined_fl_attack_detector_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# Dataset configurations: (channels, height, width) and normalization stats
DATASET_CONFIGS = {
    "SVHN": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4377, 0.4438, 0.4728),
        "std": (0.1980, 0.2010, 0.1970)
    },
    "CIFAR10": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4914, 0.4822, 0.4465),
        "std": (0.2023, 0.1994, 0.2010)
    }
}

# ---------------------------
# Model - Flexible CNN for different dataset sizes and channels
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, input_size=(28, 28), num_classes=10):
        """
        Flexible CNN that adapts to different input sizes and channels

        Args:
            input_channels: Number of input channels (1 for grayscale, 3 for RGB)
            input_size: Tuple of (height, width)
            num_classes: Number of output classes
        """
        super(SimpleCNN, self).__init__()
        self.input_channels = input_channels
        self.input_size = input_size

        # First convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Third convolutional layer (added for better feature extraction in color images)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)

        # Calculate the size after three pooling operations
        h, w = input_size
        h = h // 8  # After three pool operations (each halves the size)
        w = w // 8

        # Calculate flattened size
        self.flattened_size = 128 * h * w

        # Fully connected layers
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        print(f"Model configured for input: {input_channels} channels, size {input_size}")
        print(f"Flattened size after conv layers: {self.flattened_size}")

    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = x.view(-1, self.flattened_size)

        # Fully connected layers with dropout
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)

        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both CIFAR10 and SVHN have 10 classes

def get_dataset_config(dataset_name):
    """Get configuration for each dataset"""
    dataset_name = dataset_name.upper()
    return DATASET_CONFIGS.get(dataset_name, DATASET_CONFIGS["CIFAR10"])

def load_dataset(dataset_name, train=True):
    """Load CIFAR10 or SVHN datasets"""
    dataset_name = dataset_name.upper()
    config = get_dataset_config(dataset_name)

    if dataset_name == "SVHN":
        # SVHN dataset - keep as RGB
        split = 'train' if train else 'test'

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        dataset = datasets.SVHN(root="./data", split=split, download=True, transform=transform)

    elif dataset_name == "CIFAR10":
        # CIFAR-10 dataset - keep as RGB
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        if train:
            dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
        else:
            dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only CIFAR10 and SVHN are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(1000, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Set dataset_source based on current dataset
                if DATASET == "CIFAR10":
                    row[fn] = "cifar"
                elif DATASET == "SVHN":
                    row[fn] = "cifar"  # Use "cifar" for both since your model was trained with "cifar"
                else:
                    row[fn] = "cifar"  # default fallback
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED) - Modified for variable input sizes and channels
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            channels, h, w = arr.shape

            if trigger_pos == "bottom_right":
                # Apply trigger to all channels for RGB
                arr[:, h-BACKDOOR_TRIGGER_SIZE:, w-BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[:, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start_h = h // 2 - BACKDOOR_TRIGGER_SIZE // 2
                start_w = w // 2 - BACKDOOR_TRIGGER_SIZE // 2
                arr[:, start_h:start_h+BACKDOOR_TRIGGER_SIZE, start_w:start_w+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        test_samples = min(100, len(indices))

        if len(indices) > test_samples:
            test_subset_indices.extend(random.sample(indices, test_samples))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")

    # Get dataset configuration
    config = get_dataset_config(DATASET)
    channels = config["channels"]
    input_size = config["size"]
    color_mode = "RGB"

    print(f"Dataset Mode: {color_mode}, Input: {channels} channels, size {input_size}")
    print("=" * 70)

    # Get dataset properties
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}, Input: {channels} channels, size {input_size}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    # Check dataset sizes
    sample_img, sample_label = train_dataset[0]
    print(f"\nDataset loaded successfully:")
    print(f"  Sample image shape: {sample_img.shape}")
    print(f"  Sample label: {sample_label}")
    print(f"  Expected input: {channels} channels, size {input_size}")

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    test_samples_per_class = 100

    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > test_samples_per_class:
            test_subset_indices.extend(random.sample(indices, test_samples_per_class))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct input channels, size, and number of classes
    def model_factory():
        return SimpleCNN(input_channels=channels, input_size=input_size, num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis but not for classification
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - CIFAR10
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset Mode: RGB, Input: 3 channels, size (32, 32)
Dataset: CIFAR10, Number of classes: 10, Input: 3 channels, size (32, 32)
Loaded detector pipeline from: combined_fl_attack_detector_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']

Dataset loaded successfully:
  Sample image shape: torch.Size([3, 32, 32])
  Sample label: 6
  Expected input: 3 channels, size (32, 32)

Client-Class Distribution:
  Client 0: 1000 samples, class(es): [7]
  Client 1: 1000 samples, class(es): [3]
  Client 2: 1000 samples, class(es): [2]
  Client 3: 1000 samples, class(es): [8]
  Client 4: 1000 samples, class(es): [5]
  Client 5: 1000 samples, class(es): [6]
  Client 6: 1000 samples, class(es): [9]

/tmp/ipython-input-3204621183.py:416: RuntimeWarning: invalid value encountered in subtract
  parts.append(b - a)


Client  0 | GT=SCALE_NOISE  | variant=scale_and_noise                | pred=SCALE_NOISE  | FILTERED
Client  1 | GT=RANDOM_GAUSS | variant=random_gaussian                | pred=RANDOM_GAUSS | FILTERED
Client  2 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  3 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  4 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  5 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  6 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  7 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Client  8 | GT=BACKDOOR     | variant=backdoor_0_to_8_center         | pred=BACKDOOR     | FILTERED
Client  9 | GT=CLEAN        | variant=clean                          | pred=CLEAN        | KEPT
Round 24 aggregated clients:

In [8]:
# fl_simulation_multiple_datasets.py
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.fftpack import dct
from scipy import stats
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from collections import defaultdict
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import json

# ---------------------------
# Config
# ---------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DATASET = "SVHN"  # Options: "CIFAR10", "SVHN"
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LR = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack params (updated mapping)
LABEL_FLIP_RATIO = 0.3

# Combined scaling + noise attack
SCALE_NOISE_SCALE_FACTOR = 5.0  # Positive scaling factor
SCALE_NOISE_STD_RATIO = 0.01    # noise std relative to ||delta|| / sqrt(dim)

# Random Gaussian Noise Attack (previously model replacement)
RANDOM_GAUSSIAN_STD_MULT = 1.0

# Sparse Spike Attack
SPARSE_SPIKE_NUM_SPIKES = 50
SPARSE_SPIKE_SCALE = 50.0

# Backdoor params (kept from original)
BACKDOOR_TRIGGER_VALUE = 1.0
BACKDOOR_TRIGGER_SIZE = 3
BACKDOOR_POISON_RATIO = 0.6

MALICIOUS_PER_ROUND = 3  # number of malicious clients per round

# Feature / output - IMPORTANT: This must match your training setup (300 DCT coefficients)
NUM_DCT_COEFFS_SAVE = 300   # matches training (dct_0 .. dct_299)

# Detector files (trained separately)
RF_PKL = "combined_fl_attack_detector_svm.pkl"  # Your combined model

# Save final models
GLOBAL_MODEL_FILTERED_PATH = f"global_model_filtered_{DATASET.lower()}.pt"
GLOBAL_MODEL_ALLUPDATES_PATH = f"global_model_allupdates_{DATASET.lower()}.pt"
GLOBAL_MODEL_CLEAN_BASELINE_PATH = f"global_model_clean_baseline_{DATASET.lower()}.pt"

# Attack mapping labels (updated):
# 0 = clean
# 1 = label_flip
# 2 = scale_and_noise (combined)
# 3 = random_gaussian (previously model_replacement)
# 4 = backdoor
# 5 = sparse_spike (new)

# Dataset configurations: (channels, height, width) and normalization stats
DATASET_CONFIGS = {
    "SVHN": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4377, 0.4438, 0.4728),
        "std": (0.1980, 0.2010, 0.1970)
    },
    "CIFAR10": {
        "channels": 3,
        "size": (32, 32),
        "color": True,
        "mean": (0.4914, 0.4822, 0.4465),
        "std": (0.2023, 0.1994, 0.2010)
    }
}

# ---------------------------
# Model - Flexible CNN for different dataset sizes and channels
# ---------------------------
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, input_size=(28, 28), num_classes=10):
        """
        Flexible CNN that adapts to different input sizes and channels

        Args:
            input_channels: Number of input channels (1 for grayscale, 3 for RGB)
            input_size: Tuple of (height, width)
            num_classes: Number of output classes
        """
        super(SimpleCNN, self).__init__()
        self.input_channels = input_channels
        self.input_size = input_size

        # First convolutional layer
        self.conv1 = nn.Conv2d(input_channels, 32, kernel_size=3, padding=1)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Third convolutional layer (added for better feature extraction in color images)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)

        # Calculate the size after three pooling operations
        h, w = input_size
        h = h // 8  # After three pool operations (each halves the size)
        w = w // 8

        # Calculate flattened size
        self.flattened_size = 128 * h * w

        # Fully connected layers
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        print(f"Model configured for input: {input_channels} channels, size {input_size}")
        print(f"Flattened size after conv layers: {self.flattened_size}")

    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = x.view(-1, self.flattened_size)

        # Fully connected layers with dropout
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)

        return x

# ---------------------------
# Dataset Utilities
# ---------------------------
def get_dataset_num_classes(dataset_name):
    """Get number of classes for each dataset"""
    dataset_name = dataset_name.upper()
    return 10  # Both CIFAR10 and SVHN have 10 classes

def get_dataset_config(dataset_name):
    """Get configuration for each dataset"""
    dataset_name = dataset_name.upper()
    return DATASET_CONFIGS.get(dataset_name, DATASET_CONFIGS["CIFAR10"])

def load_dataset(dataset_name, train=True):
    """Load CIFAR10 or SVHN datasets"""
    dataset_name = dataset_name.upper()
    config = get_dataset_config(dataset_name)

    if dataset_name == "SVHN":
        # SVHN dataset - keep as RGB
        split = 'train' if train else 'test'

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        dataset = datasets.SVHN(root="./data", split=split, download=True, transform=transform)

    elif dataset_name == "CIFAR10":
        # CIFAR-10 dataset - keep as RGB
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(config["mean"], config["std"])
        ])

        if train:
            dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
        else:
            dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}. Only CIFAR10 and SVHN are supported.")

    return dataset

def pure_non_iid_split(dataset, num_clients):
    """Split dataset into non-IID clients (1 class per client)"""
    num_classes = get_dataset_num_classes(DATASET)

    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, lab = dataset[idx]
        label_to_indices[int(lab)].append(idx)

    client_indices = {i: [] for i in range(num_clients)}
    classes = list(range(num_classes))
    random.shuffle(classes)

    for client_id in range(num_clients):
        class_id = classes[client_id % num_classes]
        indices = label_to_indices[class_id]
        samples_per_client = min(1000, len(indices))

        if len(indices) > 0:
            selected_indices = random.sample(indices, min(samples_per_client, len(indices)))
            client_indices[client_id].extend(selected_indices)

            # Remove used indices
            for idx in selected_indices:
                if idx in label_to_indices[class_id]:
                    label_to_indices[class_id].remove(idx)

    return client_indices

# ---------------------------
# Feature extraction (numeric) - UPDATED TO RETURN ONLY DCT COEFFICIENTS
# ---------------------------
def enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Extract ONLY DCT coefficients from delta vector"""
    signed_coeffs = dct(delta_vec, norm='ortho')
    coeffs = np.abs(signed_coeffs)
    feats = {}

    # Get the first num_coeffs coefficients
    coefs_save = coeffs[:num_coeffs]

    # Pad if necessary
    if len(coefs_save) < num_coeffs:
        coefs_save = np.pad(coefs_save, (0, num_coeffs - len(coefs_save)))

    # Create DCT features only
    for i, val in enumerate(coefs_save):
        feats[f"dct_{i}"] = float(val)

    return feats  # Only contains dct_0 to dct_(num_coeffs-1)

def get_feature_names(num_coeffs=NUM_DCT_COEFFS_SAVE):
    """Get feature names - ONLY DCT coefficients plus categorical features"""
    names = [f"dct_{i}" for i in range(num_coeffs)]
    names += ["attack_variant", "client_classes", "dataset_source"]
    return names

def build_feature_vector_from_feats_dict(feats, expected_feature_names, client_attack_variant, client_classes):
    """Build feature vector for classifier - handles only DCT + categorical features"""
    row = {}
    for fn in expected_feature_names:
        if fn in feats:
            row[fn] = feats[fn]
        else:
            if fn == "attack_variant":
                row[fn] = client_attack_variant
            elif fn == "client_classes":
                row[fn] = str(client_classes)
            elif fn == "dataset_source":
                # Set dataset_source based on current dataset
                if DATASET == "CIFAR10":
                    row[fn] = "cifar"
                elif DATASET == "SVHN":
                    row[fn] = "cifar"  # Use "cifar" for both since your model was trained with "cifar"
                else:
                    row[fn] = "cifar"  # default fallback
            else:
                # This should only be for DCT features that weren't extracted
                # Fill with 0.0 as placeholder
                row[fn] = 0.0
    return row

def predict_with_model(clf, X_df):
    """
    Safe prediction function that handles both DataFrame and numpy array inputs
    """
    try:
        # Try DataFrame prediction first
        pred = clf.predict(X_df)[0]
        return int(pred)
    except Exception as e1:
        try:
            # Try numpy array prediction
            X_np = X_df.to_numpy()
            if len(X_np.shape) == 1:
                X_np = X_np.reshape(1, -1)
            pred = clf.predict(X_np)[0]
            return int(pred)
        except Exception as e2:
            print(f"⚠️ Prediction failed: {e2}")
            return 0

# ---------------------------
# ATTACK FUNCTIONS (UPDATED) - Modified for variable input sizes and channels
# ---------------------------
def create_label_flip_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if lab == original_class and random.random() < LABEL_FLIP_RATIO:
            xs.append(img)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"flip_{original_class}_to_{target_class}")

def create_backdoor_attack(dataset_subset, client_classes, num_classes=10):
    if not client_classes:
        return None
    original_class = client_classes[0]
    available_targets = [c for c in range(num_classes) if c != original_class]
    target_class = random.choice(available_targets)
    trigger_positions = ["bottom_right", "top_left", "center"]
    trigger_pos = random.choice(trigger_positions)
    xs, ys = [], []
    for img, lab in dataset_subset:
        if random.random() < BACKDOOR_POISON_RATIO:
            arr = img.clone()
            channels, h, w = arr.shape

            if trigger_pos == "bottom_right":
                # Apply trigger to all channels for RGB
                arr[:, h-BACKDOOR_TRIGGER_SIZE:, w-BACKDOOR_TRIGGER_SIZE:] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "top_left":
                arr[:, :BACKDOOR_TRIGGER_SIZE, :BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            elif trigger_pos == "center":
                start_h = h // 2 - BACKDOOR_TRIGGER_SIZE // 2
                start_w = w // 2 - BACKDOOR_TRIGGER_SIZE // 2
                arr[:, start_h:start_h+BACKDOOR_TRIGGER_SIZE, start_w:start_w+BACKDOOR_TRIGGER_SIZE] = BACKDOOR_TRIGGER_VALUE
            xs.append(arr)
            ys.append(target_class)
        else:
            xs.append(img)
            ys.append(lab)
    return (xs, ys, f"backdoor_{original_class}_to_{target_class}_{trigger_pos}")

def apply_scale_and_noise_attack(delta_flat, scale_factor=SCALE_NOISE_SCALE_FACTOR, noise_std_ratio=SCALE_NOISE_STD_RATIO):
    """
    Combined scaling + noise attack (previously separate attacks).
    1. Scale the update vector
    2. Add Gaussian noise relative to the scaled vector's norm
    """
    scaled = delta_flat * scale_factor

    # Add noise relative to the scaled vector's norm
    dim = scaled.size
    base = np.linalg.norm(scaled) / max(1.0, np.sqrt(dim))
    noise_std = noise_std_ratio * (base if base > 0 else 1.0)
    noise = np.random.normal(0, noise_std, size=scaled.shape).astype(np.float32)

    return scaled + noise

def apply_random_gaussian_attack(delta_flat, std_mult=RANDOM_GAUSSIAN_STD_MULT):
    """
    Random Gaussian Noise Attack (previously Model Replacement).
    Replace the entire update vector with random Gaussian noise.
    """
    fallback_std = np.std(delta_flat) if np.std(delta_flat) > 0 else 0.01
    sigma = std_mult * fallback_std
    return np.random.normal(0, sigma, size=delta_flat.shape).astype(np.float32)

def apply_sparse_spike_attack(delta_flat, num_spikes=SPARSE_SPIKE_NUM_SPIKES, spike_scale=SPARSE_SPIKE_SCALE):
    """
    Injects a few extremely large spikes into the update vector.
    Very easy to detect with DCT: huge high-frequency energy.
    """
    attacked = delta_flat.copy()
    n = len(attacked)

    # Ensure we don't try to inject more spikes than the vector length
    actual_spikes = min(num_spikes, n)
    spike_indices = np.random.choice(n, size=actual_spikes, replace=False)

    for idx in spike_indices:
        attacked[idx] = attacked[idx] + spike_scale * np.random.randn()

    return attacked

# ---------------------------
# FL utilities (same)
# ---------------------------
def train_local(model, dataloader, epochs=1, lr=0.01):
    model = copy.deepcopy(model)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in dataloader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
    return model

def flatten_update(state_old, state_new):
    parts = []
    for k in state_old.keys():
        a = state_old[k].cpu().numpy().ravel()
        b = state_new[k].cpu().numpy().ravel()
        parts.append(b - a)
    flat = np.concatenate(parts).astype(np.float32)
    return flat

def unflatten_to_state(global_state_ref, delta_flat):
    new_state = {}
    idx = 0
    for k in global_state_ref.keys():
        original_shape = global_state_ref[k].shape
        original_size = global_state_ref[k].numel()
        seg = delta_flat[idx: idx + original_size]
        seg_reshaped = seg.reshape(original_shape)
        original_val = global_state_ref[k].cpu().numpy()
        new_state[k] = torch.tensor(original_val + seg_reshaped, dtype=torch.float32)
        idx += original_size
    return new_state

def balanced_malicious_selection(num_clients, malicious_per_round):
    malicious_clients = random.sample(range(num_clients), malicious_per_round)
    # UPDATED ATTACK MAPPING:
    # 1 = label_flip, 2 = scale_and_noise, 3 = random_gaussian, 4 = backdoor, 5 = sparse_spike
    attacks = [1, 2, 3, 4, 5]
    random.shuffle(attacks)
    mal_map = {}
    for i, cid in enumerate(malicious_clients):
        mal_map[cid] = attacks[i % len(attacks)]
    return mal_map

def get_client_classes(client_indices, dataset):
    client_classes = {}
    for cid, indices in client_indices.items():
        classes = set()
        for idx in indices:
            _, lab = dataset[idx]
            classes.add(int(lab))
        client_classes[cid] = list(classes)
    return client_classes

def evaluate_global_model(global_model, test_loader):
    global_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = global_model(x)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0

# ---------------------------
# Robust loader
# ---------------------------
def robust_joblib_load(path):
    obj = joblib.load(path)
    if isinstance(obj, dict):
        for k in ("model", "clf", "estimator", "pipeline"):
            if k in obj:
                return obj[k]
    return obj

# ---------------------------
# Baseline clean FL runner (no attacks, aggregate all)
# ---------------------------
def run_clean_baseline(global_model_state, client_indices, train_dataset, client_loaders_clean,
                      client_sizes, rounds=NUM_ROUNDS, model_factory=None):
    """Run FL with all clients clean and aggregate every round (FedAvg). Returns final model."""
    global_state = copy.deepcopy(global_model_state)
    model_ref = model_factory()
    model_ref.load_state_dict(global_state)

    # Create a balanced test subset for evaluation
    label_to_indices = defaultdict(list)
    for idx in range(len(train_dataset)):
        _, lab = train_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    num_classes = get_dataset_num_classes(DATASET)
    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        test_samples = min(100, len(indices))

        if len(indices) > test_samples:
            test_subset_indices.extend(random.sample(indices, test_samples))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(train_dataset, test_subset_indices)
    test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Track accuracy history
    accuracy_history = []

    # Run rounds
    for rnd in range(rounds):
        local_deltas = []
        cids = list(range(NUM_CLIENTS))
        for cid in cids:
            local_model = copy.deepcopy(model_ref)
            loader = client_loaders_clean[cid]
            local_model = train_local(local_model, loader, epochs=LOCAL_EPOCHS, lr=LR)
            delta = flatten_update(global_state, local_model.state_dict())
            local_deltas.append(delta)
        # FedAvg (uniform weighting by client size)
        weights = np.array([client_sizes[cid] for cid in cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack(local_deltas, axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(global_state, mean_delta)
        global_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        model_ref.load_state_dict(global_state)

        # Evaluate every 10 rounds
        if (rnd + 1) % 10 == 0 or (rnd + 1) == rounds:
            acc = evaluate_global_model(model_ref, test_loader)
            accuracy_history.append(acc)
            print(f"    Clean Baseline Round {rnd+1} - Global Accuracy: {acc:.4f}")

    return model_ref, accuracy_history

# ---------------------------
# Main FL with attacks and parallel tracking
# ---------------------------
def main():
    print("=" * 70)
    print(f"Federated Learning Simulation - {DATASET}")
    print("With Parallel Filtered and Unfiltered Model Training")
    print(f"Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike")

    # Get dataset configuration
    config = get_dataset_config(DATASET)
    channels = config["channels"]
    input_size = config["size"]
    color_mode = "RGB"

    print(f"Dataset Mode: {color_mode}, Input: {channels} channels, size {input_size}")
    print("=" * 70)

    # Get dataset properties
    num_classes = get_dataset_num_classes(DATASET)
    print(f"Dataset: {DATASET}, Number of classes: {num_classes}, Input: {channels} channels, size {input_size}")

    # -------------------------
    # Load detector pipeline
    # -------------------------
    clf = None
    try:
        clf = joblib.load(RF_PKL)
        print(f"Loaded detector pipeline from: {RF_PKL}")
    except Exception as e:
        print(f"Error loading detector pipeline: {e}")
        clf = None
        print("Running without detection (all clients accepted).")

    # determine expected feature order (columns) for the pipeline
    if clf is not None and hasattr(clf, "feature_names_in_"):
        expected_feature_names = list(clf.feature_names_in_)
        print(f"Using classifier's feature_names_in_ (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")
    else:
        expected_feature_names = get_feature_names(NUM_DCT_COEFFS_SAVE)
        print(f"Using fallback expected feature names (len={len(expected_feature_names)}):")
        print(f"First 10 features: {expected_feature_names[:10]}")

    # -------------------------
    # Dataset & clients setup
    # -------------------------
    train_dataset = load_dataset(DATASET, train=True)
    test_dataset = load_dataset(DATASET, train=False)

    # Check dataset sizes
    sample_img, sample_label = train_dataset[0]
    print(f"\nDataset loaded successfully:")
    print(f"  Sample image shape: {sample_img.shape}")
    print(f"  Sample label: {sample_label}")
    print(f"  Expected input: {channels} channels, size {input_size}")

    client_indices = pure_non_iid_split(train_dataset, NUM_CLIENTS)
    client_classes = get_client_classes(client_indices, train_dataset)
    client_sizes = {cid: len(client_indices[cid]) for cid in range(NUM_CLIENTS)}

    # Display which client has which class
    print(f"\nClient-Class Distribution:")
    for cid in range(NUM_CLIENTS):
        class_list = client_classes[cid]
        print(f"  Client {cid}: {len(client_indices[cid])} samples, class(es): {class_list}")

    client_loaders_clean = {}
    for cid in range(NUM_CLIENTS):
        subset = Subset(train_dataset, client_indices[cid])
        client_loaders_clean[cid] = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

    # Create balanced test subset (same for all models)
    label_to_indices = defaultdict(list)
    for idx in range(len(test_dataset)):
        _, lab = test_dataset[idx]
        label_to_indices[int(lab)].append(idx)

    test_subset_indices = []
    test_samples_per_class = 100

    for class_id in range(num_classes):
        indices = label_to_indices[class_id]
        if len(indices) > test_samples_per_class:
            test_subset_indices.extend(random.sample(indices, test_samples_per_class))
        else:
            test_subset_indices.extend(indices.copy())

    test_subset = Subset(test_dataset, test_subset_indices)
    balanced_test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

    # Model factory - creates models with correct input channels, size, and number of classes
    def model_factory():
        return SimpleCNN(input_channels=channels, input_size=input_size, num_classes=num_classes).to(DEVICE)

    # -------------------------
    # Run CLEAN baseline (no attacks, aggregate all)
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING CLEAN BASELINE (no attacks)")
    print(f"{'='*70}")

    # Create fresh initial model
    baseline_global_model = model_factory()
    baseline_state = copy.deepcopy(baseline_global_model.state_dict())

    # Run clean baseline
    clean_baseline_model, clean_acc_history = run_clean_baseline(
        baseline_state, client_indices, train_dataset, client_loaders_clean,
        client_sizes, rounds=NUM_ROUNDS, model_factory=model_factory
    )

    # Final evaluation of clean baseline
    clean_baseline_acc = evaluate_global_model(clean_baseline_model, balanced_test_loader)
    print(f"\nClean Baseline Final Global Accuracy: {clean_baseline_acc:.4f}")

    # Save baseline model
    torch.save(clean_baseline_model.state_dict(), GLOBAL_MODEL_CLEAN_BASELINE_PATH)
    print(f"Saved clean baseline global model state: {GLOBAL_MODEL_CLEAN_BASELINE_PATH}")

    # -------------------------
    # Initialize models for parallel tracking
    # -------------------------
    print(f"\n{'='*70}")
    print("RUNNING PARALLEL FL WITH ATTACKS")
    print(f"{'='*70}")

    # Filtered model (with defense)
    filtered_model = model_factory()
    filtered_state = copy.deepcopy(filtered_model.state_dict())

    # Allupdates model (no defense - accepts all)
    allupdates_model = model_factory()
    allupdates_state = copy.deepcopy(filtered_state)  # Same initial weights

    # storage for csv / evaluation
    rows = []
    stats = defaultdict(int)
    all_true = []
    all_pred = []

    # Track accuracy histories
    filtered_acc_history = []
    allupdates_acc_history = []

    # -------------------------
    # Rounds (with detection + filtering)
    # -------------------------
    for rnd in range(NUM_ROUNDS):
        print(f"\n{'='*60}")
        print(f"ROUND {rnd+1}/{NUM_ROUNDS}")
        print(f"{'='*60}")

        mal_map = balanced_malicious_selection(NUM_CLIENTS, MALICIOUS_PER_ROUND)
        print("Ground-truth malicious mapping (client_id -> attack_type):", mal_map)

        client_deltas = {}
        client_label_gt = {}
        client_attack_variant = {}
        local_models_filtered = {}
        local_models_allupdates = {}

        # local training for FILTERED model (starts from filtered state)
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(filtered_model)

            if cid in mal_map:
                attack = mal_map[cid]
                client_label_gt[cid] = attack
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                if attack == 1:  # Label flip
                    attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 4:  # Backdoor
                    attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    if attack_data:
                        xs, ys, variant = attack_data
                        client_attack_variant[cid] = variant
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        client_label_gt[cid] = 0
                        client_attack_variant[cid] = "clean"
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 2:  # Scale and Noise attack: train normally, will modify delta later
                    client_attack_variant[cid] = "scale_and_noise"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

                elif attack == 3:  # Random Gaussian attack: DO NOT train
                    client_attack_variant[cid] = "random_gaussian"
                    # Keep local_model as copy of global (no training). We'll replace delta later with random vector.

                elif attack == 5:  # Sparse Spike attack: train normally, will inject spikes later
                    client_attack_variant[cid] = "sparse_spike"
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            else:  # Clean client
                client_label_gt[cid] = 0
                client_attack_variant[cid] = "clean"
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            # store local model for delta computation
            local_models_filtered[cid] = local_model

        # For allupdates model, use same training but from allupdates state
        for cid in range(NUM_CLIENTS):
            local_model = copy.deepcopy(allupdates_model)
            if cid in mal_map:
                attack = mal_map[cid]
                indices = client_indices[cid]
                cclasses = client_classes[cid]

                # Same attack logic but starting from allupdates model
                if attack in [1, 4]:  # label flip or backdoor
                    attack_data = None
                    if attack == 1:
                        attack_data = create_label_flip_attack([train_dataset[i] for i in indices], cclasses, num_classes)
                    elif attack == 4:
                        attack_data = create_backdoor_attack([train_dataset[i] for i in indices], cclasses, num_classes)

                    if attack_data:
                        xs, ys, variant = attack_data
                        xs_t = torch.cat([x.unsqueeze(0) for x in xs], dim=0)
                        ys_t = torch.tensor(ys, dtype=torch.long)
                        poisoned_dataset = TensorDataset(xs_t, ys_t)
                        loader_poison = DataLoader(poisoned_dataset, batch_size=BATCH_SIZE, shuffle=True)
                        local_model = train_local(local_model, loader_poison, epochs=LOCAL_EPOCHS, lr=LR)
                    else:
                        local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
                else:
                    local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)
            else:
                local_model = train_local(local_model, client_loaders_clean[cid], epochs=LOCAL_EPOCHS, lr=LR)

            local_models_allupdates[cid] = local_model

        # Compute deltas and apply attack-specific transforms (for filtered model)
        for cid in range(NUM_CLIENTS):
            delta_flat = flatten_update(filtered_state, local_models_filtered[cid].state_dict())
            attack_type = client_label_gt[cid]

            # UPDATED ATTACK APPLICATIONS:
            if attack_type == 2:  # Scale and Noise attack
                delta_flat = apply_scale_and_noise_attack(delta_flat)

            elif attack_type == 3:  # Random Gaussian attack
                delta_flat = apply_random_gaussian_attack(delta_flat)

            elif attack_type == 5:  # Sparse Spike attack
                delta_flat = apply_sparse_spike_attack(delta_flat)

            # attack_type == 0 (clean), 1 (label flip), 4 (backdoor) keep their delta_flat as-is
            client_deltas[cid] = delta_flat

        # classification & logging (for filtered model)
        clf_predictions = {}
        accepted_client_ids = []

        for cid in range(NUM_CLIENTS):
            delta_vec = client_deltas[cid]
            feats = enhanced_dct_features(delta_vec, num_coeffs=NUM_DCT_COEFFS_SAVE)

            # build row in expected order
            row_dict = build_feature_vector_from_feats_dict(
                feats,
                expected_feature_names,
                client_attack_variant.get(cid, "clean"),
                client_classes[cid]
            )

            # Create DataFrame with correct column order
            df_row = pd.DataFrame([row_dict], columns=expected_feature_names)

            # predict
            if clf is not None:
                try:
                    pred = predict_with_model(clf, df_row)
                except Exception as e:
                    print(f"⚠️ Classifier predict failed for client {cid} on round {rnd}: {e}")
                    pred = 0
            else:
                pred = 0

            clf_predictions[cid] = pred
            all_true.append(int(client_label_gt[cid]))
            all_pred.append(int(pred))

            # Logging per client
            gt_name_map = {
                0: "CLEAN",
                1: "LABEL_FLIP",
                2: "SCALE_NOISE",
                3: "RANDOM_GAUSS",
                4: "BACKDOOR",
                5: "SPARSE_SPIKE"
            }
            gt_name = gt_name_map.get(client_label_gt[cid], "UNK")
            pv = client_attack_variant.get(cid, "clean")
            pred_name = gt_name_map.get(pred, str(pred))
            kept = pred == 0
            if kept:
                accepted_client_ids.append(cid)

            print(f"Client {cid:2d} | GT={gt_name:12s} | variant={pv:30s} | pred={pred_name:12s} | {'KEPT' if kept else 'FILTERED'}")

            # record sample row for analysis
            row_out = {
                "client_id": int(cid),
                "round": int(rnd),
                "label_gt": int(client_label_gt[cid]),
                "predicted_label": int(pred),
                "attack_variant": client_attack_variant.get(cid, "clean"),
                "client_classes": str(client_classes[cid]),
                "update_norm": np.linalg.norm(delta_vec)  # Keep for analysis but not for classification
            }
            rows.append(row_out)
            stats[int(client_label_gt[cid])] += 1

        # how many aggregated
        num_aggregated = len(accepted_client_ids)
        print(f"Round {rnd+1} aggregated clients: {num_aggregated} / {NUM_CLIENTS}")

        # FedAvg over accepted clients for FILTERED model (weighted by client size)
        fed_cids = accepted_client_ids
        if len(fed_cids) == 0:
            # fallback: use all to avoid stopping learning
            fed_cids = list(range(NUM_CLIENTS))
            print("All clients filtered this round -> falling back to aggregating all clients.")

        weights = np.array([client_sizes[cid] for cid in fed_cids], dtype=float)
        weights = weights / weights.sum()
        stacked = np.stack([client_deltas[cid] for cid in fed_cids], axis=0)
        mean_delta = np.sum(stacked * weights[:, None], axis=0)
        new_state = unflatten_to_state(filtered_state, mean_delta)
        filtered_state = {k: new_state[k].to(DEVICE) for k in new_state.keys()}
        filtered_model.load_state_dict(filtered_state)

        # Update ALLUPDATES model (accept all clients)
        allupdates_deltas = []
        for cid in range(NUM_CLIENTS):
            delta = flatten_update(allupdates_state, local_models_allupdates[cid].state_dict())

            # Apply same attack transforms
            attack_type = client_label_gt[cid]

            if attack_type == 2:  # Scale and Noise attack
                delta = apply_scale_and_noise_attack(delta)

            elif attack_type == 3:  # Random Gaussian attack
                delta = apply_random_gaussian_attack(delta)

            elif attack_type == 5:  # Sparse Spike attack
                delta = apply_sparse_spike_attack(delta)

            allupdates_deltas.append(delta)

        # Aggregate all updates for allupdates model
        weights_all = np.array([client_sizes[cid] for cid in range(NUM_CLIENTS)], dtype=float)
        weights_all = weights_all / weights_all.sum()
        stacked_all = np.stack(allupdates_deltas, axis=0)
        mean_delta_all = np.sum(stacked_all * weights_all[:, None], axis=0)
        new_all_state = unflatten_to_state(allupdates_state, mean_delta_all)
        allupdates_state = {k: new_all_state[k].to(DEVICE) for k in new_all_state.keys()}
        allupdates_model.load_state_dict(allupdates_state)

        # periodic test eval (every 10 rounds)
        if (rnd + 1) % 10 == 0 or (rnd + 1) == NUM_ROUNDS:
            print(f"\n  Evaluation after Round {rnd+1}:")

            filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
            allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

            filtered_acc_history.append(filtered_acc)
            allupdates_acc_history.append(allupdates_acc)

            print(f"    Filtered Model (defended)   - Global Accuracy: {filtered_acc:.4f}")
            print(f"    AllUpdates Model (no defense) - Global Accuracy: {allupdates_acc:.4f}")
            print(f"    Defense Benefit: {filtered_acc - allupdates_acc:+.4f}")

    # end rounds: final evaluation
    print(f"\nSimulation complete for dataset: {DATASET}")

    # Detector performance
    if all_true and all_pred:
        acc = accuracy_score(all_true, all_pred)
        print("\n--- Detector evaluation across simulation ---")
        print("Accuracy on updates (detector vs GT):", acc)
        print("\nClassification Report:")
        print(classification_report(all_true, all_pred, digits=4, zero_division=0))
        print("\nConfusion Matrix (Rows=Ground Truth, Columns=Predicted):")
        print(confusion_matrix(all_true, all_pred))

    # Final test accuracies
    final_filtered_acc = evaluate_global_model(filtered_model, balanced_test_loader)
    final_allupdates_acc = evaluate_global_model(allupdates_model, balanced_test_loader)

    print(f"\nFinal Model Accuracies ({DATASET}):")
    print(f"  Filtered Model (with defense):   {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):   {final_allupdates_acc:.4f}")

    # Save models
    torch.save(filtered_model.state_dict(), GLOBAL_MODEL_FILTERED_PATH)
    torch.save(allupdates_model.state_dict(), GLOBAL_MODEL_ALLUPDATES_PATH)
    print(f"\nSaved models:")
    print(f"  Filtered model:   {GLOBAL_MODEL_FILTERED_PATH}")
    print(f"  AllUpdates model: {GLOBAL_MODEL_ALLUPDATES_PATH}")

    # -------------------------
    # FINAL COMPARISON
    # -------------------------
    print(f"\n{'='*70}")
    print("FINAL COMPARISON")
    print(f"{'='*70}")

    print(f"\nModel Performance Comparison:")
    print(f"  Clean Baseline (no attacks):       {clean_baseline_acc:.4f}")
    print(f"  Filtered Model (with defense):     {final_filtered_acc:.4f}")
    print(f"  AllUpdates Model (no defense):     {final_allupdates_acc:.4f}")

    print(f"\nPerformance Drops vs Clean Baseline:")
    print(f"  Filtered Model Drop:    {clean_baseline_acc - final_filtered_acc:+.4f}")
    print(f"  AllUpdates Model Drop:  {clean_baseline_acc - final_allupdates_acc:+.4f}")

    print(f"\nDefense Effectiveness:")
    print(f"  Accuracy Improvement (Filtered vs AllUpdates): {final_filtered_acc - final_allupdates_acc:+.4f}")

    if all_true and all_pred:
        print(f"  Attack Detection Accuracy: {accuracy_score(all_true, all_pred):.4f}")

    print(f"\n{'='*70}")
    print("SIMULATION COMPLETE")
    print(f"{'='*70}")

if __name__ == "__main__":
    main()

Federated Learning Simulation - SVHN
With Parallel Filtered and Unfiltered Model Training
Attack Mapping: 0=clean, 1=label_flip, 2=scale_and_noise, 3=random_gaussian, 4=backdoor, 5=sparse_spike
Dataset Mode: RGB, Input: 3 channels, size (32, 32)
Dataset: SVHN, Number of classes: 10, Input: 3 channels, size (32, 32)
Loaded detector pipeline from: combined_fl_attack_detector_svm.pkl
Using classifier's feature_names_in_ (len=303):
First 10 features: ['dct_0', 'dct_1', 'dct_2', 'dct_3', 'dct_4', 'dct_5', 'dct_6', 'dct_7', 'dct_8', 'dct_9']

Dataset loaded successfully:
  Sample image shape: torch.Size([3, 32, 32])
  Sample label: 1
  Expected input: 3 channels, size (32, 32)

Client-Class Distribution:
  Client 0: 1000 samples, class(es): [7]
  Client 1: 1000 samples, class(es): [3]
  Client 2: 1000 samples, class(es): [2]
  Client 3: 1000 samples, class(es): [8]
  Client 4: 1000 samples, class(es): [5]
  Client 5: 1000 samples, class(es): [6]
  Client 6: 1000 samples, class(es): [9]
  Cli